# 3D-spectrospherics : why it's not straightforward

The problem is that... magnitude-phase decomposition is not *straightforwardly* applicaable in 3 dimensions. Let's see why in this notebook.

## Spherical harmonics

3d decomposition relies on a projection over a family of (real) spherical harmonics. Using $\phi$ as the angle with x axis, and $\theta$ as the angle with the z axis, the spherical harmonic of degree $\ell$ and order $m$ is defined by 

$$
Y^m_\ell(\theta, \phi) = \begin{cases}
 P^{|m|}_\ell(\cos \theta)  \sin {(|m| \phi)} \ & m \lt 0 \\
 P^m_\ell(\cos \theta)  \cos {(m \phi)} \ &  m \geq 0
\end{cases}
$$

where $P^m_\ell(x)$ is the associated Legendre polynome of order $\ell$ and degree $m$ (corresponding to ambisonics' order, tricky disambiguation here). 



In [1]:
# mixing spherical harmonics in 3 dimensions
from spectrospherics import plot_spherical_harmonic

plot_spherical_harmonic().servable()
# plot_spherical_harmonic().show() # <- in browser version

Column
    [0] Markdown(str)
    [1] Row
        [0] IntSlider(end=8, label='degree  n  (ambisonic o..., name='degree  n  (ambisonic o..., value=2)
        [1] IntSlider(end=2, label='order  m', name='order  m', start=-2, value=1)
    [2] Plotly(Figure)

In [2]:
# a field is a weighted sum of harmonics : drive the coefficients
from spectrospherics.spectrospherics import plot_spectrambisonics_3d

# plot_spectrambisonics_3d().servable()
plot_spectrambisonics_3d().show()   # <- in browser version

Launching server at http://localhost:62470


Task was destroyed but it is pending!
task: <Task pending name='Task-6536' coro=<_async_in_context.<locals>.run_in_context() done, defined at /Users/domkirke/miniconda3/envs/nngui/lib/python3.13/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-6537' coro=<Kernel.shell_main() running at /Users/domkirke/miniconda3/envs/nngui/lib/python3.13/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /Users/domkirke/miniconda3/envs/nngui/lib/python3.13/site-packages/zmq/eventloop/zmqstream.py:563]>
/Users/domkirke/miniconda3/envs/nngui/lib/python3.13/site-packages/panel/io/document.py:216: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  gc.collect()
Task was destroyed but it is pending!
task: <Task pending name='Task-6537' coro=<Kernel.shell_main() running at /Users/domkirke/miniconda3/envs/nngui/lib/python3.13/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]>


## Non-commutativity : an operational problem

Furthermore, let's have an operational approach for sound field deformation, and see what other problems it brings. 

**Euler angles.** The strength of the amplitude-phase spectrangular decomposition was that not only amplitude was rotation-invariant, but that *shifting the phase was direclty rotating the correponding circular harmonic*. Indeed, there is only one angle : a phase offset correspond to a single rotation. Clear and intuitive. 

In 3 dimensions, *it's not that simple*. Indeed, a rotation in 3 dimensions (an element of the $SO(3)$ group) can be generated by three successive rotations about coordinate axes. 
$$
\begin{align}
R(\alpha, \beta, \gamma) & = R_z(\alpha) R_y(\beta) R_z(\gamma) & \alpha, \gamma \in [0, 2 \pi ), \beta \in [0, \pi]
\end{align}
$$

where each angle has a clear geometric meaning : the pair $(\alpha, \beta)$ points the rotated pole sending each point $z$ to the direction with polar angle $\beta$ and azimuth $\alpha$, and the angle $\gamma$ being the **roll** about this direction, the extra degree of freedom that a direction does not have. For this reason, we need an Euler convention (ZYZ in the formulmation before) to identify a given rotation. 

**Non-commutativity.** $SO(3)$ is non-commutative, implying that $AB \neq BA$. Hence, **the ordering of rotations $(\alpha, \beta, \gamma)$ is essential**: 

$$
R_z(\alpha) R_y(\beta) R_z(\gamma) \neq R_z(\alpha)  R_z(\gamma) R_y(\beta)
$$


implying that if we want to have a clear *rotation* parameter, it must come from an ordered rotation of a starting field. 

**A Lie group explanation.** Lie algebra is a common algebra used to deal with non-combinatorial mathematical groups. An important notion is the Lie bracket, that measures the infinitesimal error between different pairwise orders : 

$$
[A, B] = AB - BA
$$
such that $[A, B] = - [B, A]$. With $SO(3)$, we can demostrate that
$$
\begin{align}
 [J_x, J_y] &= J_z &  [J_y, J_z] & = J_x & [J_z, J_x] & = J_y
\end{align}
$$
where $J_x, J_y, J_z$ describe infinitesimal rotations under its indicial axis. We can see that **doing a small $x$-turn then $y$, then $-y$ then $-x$, leaves a residual $z$-turn** (spin).

Equivalently, the loop $R_x(\epsilon)R_y(\delta)R_x(-\epsilon)R_y(-\delta) \approx R_z(\epsilon\delta)$ : a tiny round trip does not close, and leaves a small rotation about the third axis. See appendix B for more details. 

In [3]:
# order matters, and a round trip does not close : the Lie bracket, seen
from spectrospherics import plot_angle_noncommutativity

plot_angle_noncommutativity().servable()
# plot_angle_noncommutativity().show()   # <- in browser version

Column
    [0] Row
        [0] FloatSlider(end=180, label='ε : turn about x (°)', name='ε : turn about x (°)', step=1, value=40)
        [1] FloatSlider(end=180, label='δ : turn about y (°)', name='δ : turn about y (°)', step=1, value=40)
    [1] Plotly(Figure)
    [2] LaTeX(str, renderer='mathjax', styles={'font-size': '15px'}, width=520)
    [3] Markdown(str)

## Orbit redundency 

As we will see in *2_wigner_d_matrix.ipynb*, there is another impact of this increase in dimensionality. Taking the real spherical harmonics $\mathcal{H}_\ell$ of a given degree $\ell$, its coordinates $\mathbf{a}_\ell$ has then $2\ell + 1$ dimensions. We then have

$$
\begin{align*}
\underbrace{2\ell+1}_{\text{coefficients}}
& = \underbrace{\#\text{invariants}}_{\text{shape (rotation-fixed)}}
+ \underbrace{\dim\mathcal O}_{\text{orientation (rotation-reachable)}}
\end{align*}
$$

or

$$ 
\qquad \dim\mathcal O = 3-\dim\mathrm{Stab} \le 3.
$$

where $\mathcal{0}_{\mathbf{a}}$ is the **orbit** of a given $\mathbf{a}$ under solid rotations (the values accesible from $\mathbf{a}_\ell$ under a rotation of the field, see *B_wigner.ipynb*). This implies that up to 3 implicit orthogonal axis re-aim the same shape. If the shape is symmetric (e.g. a zonal $Y_\ell^0$), the stabilizer is non trivial, such that a rotation direction can map the shape onto itself. The knob then goes dead, and the same field is produced by an infinite continuum of rotations. 

This induces some parametrization traps, that introduce redundancy that are not in the $\mathbf{a}_\ell$: 
- *Maxwell axes* (solid angle). a degree-$\ell$ harmonic $\leftrightarrow$ an unordered set of $\ell$ axis-direction. Permuting the axes of flipping an axis with a compensating sign gives the **same coefficients** (many-to-one)
- *Angle / hyperspherical charts* : degenerate on symmetric shapes, where the Jacobian drops rank and many settings collapse to one field. 

In [4]:
# the stabilizer, felt : a rotation control that does nothing
from spectrospherics import plot_dead_knob

plot_dead_knob().servable()
# plot_dead_knob().show()   # <- in browser version

Column
    [0] Markdown(str)
    [1] Row
        [0] IntSlider(end=5, label='degree ℓ', name='degree ℓ', start=1, value=3)
        [1] Column
            [0] Markdown(str, margin=(0, 0, -10, 10))
            [1] RadioButtonGroup(button_style='outline', label='the shape', name='the shape', options=['axisymmetric  Yℓ⁰', ...], value='axisymmetric  Yℓ⁰', variant='outline', width=230)
        [2] FloatSlider(end=90, label='1. tilt the s..., name='1. tilt the s..., step=1)
    [2] Row
        [0] Column
            [0] Markdown(str, margin=(0, 0, -10, 10))
            [1] RadioButtonGroup(button_style='outline', label='knob axis', name='knob axis', options=["z — the chart's axis", ...], value="z — the chart's axis", variant='outline', width=300)
        [1] FloatSlider(end=360, label='2. turn the knob (°)', name='2. turn the knob (°)', step=1, value=90)
    [3] Row
        [0] Plotly(Figure)
        [1] Plotly(Figure)
    [4] LaTeX(str, renderer='mathjax', styles={'font-size': '15px'}, width=520)
    [5] Markdown(str)

In [5]:
# orbit, stabilizer and invariant count, measured
from spectrospherics import plot_orbit_stabilizer

plot_orbit_stabilizer().servable()
# plot_orbit_stabilizer().show()   # <- in browser version

Column
    [0] Markdown(str)
    [1] Row
        [0] IntSlider(end=5, label='degree ℓ', name='degree ℓ', start=1, value=2)
        [1] Select(label='multiplet', name='multiplet', options=['zonal  Y_ℓ⁰  (axisymmetr...], value='zonal  Y_ℓ⁰  ...)
        [2] FloatSlider(label='break the symmetry (..., name='break the symmetry (..., step=0.01)
    [2] Row
        [0] FloatSlider(end=360, label='yaw α about z (°)', name='yaw α about z (°)', step=1)
        [1] FloatSlider(end=180, label='tilt β about y (°)', name='tilt β about y (°)', step=1)
        [2] FloatSlider(end=360, label='spin γ about z, ..., name='spin γ about z, ..., step=1)
    [3] Row
        [0] Column
            [0] Plotly(Figure, config={'displayModeBar': False})
            [1] Markdown(str)
            [2] Plotly(Figure, config={'displayModeBar': False})
        [1] Plotly(Figure)
    [4] LaTeX(str, renderer='mathjax', styles={'font-size': '15px'}, width=520)
    [5] Markdown(str)

## Multi-dimensional & non-linear shape 

In 2D the shape of a mode is one number : $|c_m|$. In 3d, the sape of a degree is several invariants 

$$
\#\text{invariants}=\begin{cases}1,&\ell=0,1\\ 2\ell-2,&\ell\ge2\end{cases}
\qquad\text{and, for }\ell\ge2\ :\quad
2\ell-2 = \underbrace{1}_{\text{energy }\Vert\mathbf{a}_\ell\Vert^2}\;+\;\underbrace{2\ell-3}_{\text{shape invariants}}$$

- for $\ell=1$ there is still 1 invariant (energy), which is why degree 1 is almost 2d case
- for $\ell \geq 2$ extra shape invaiants are nonlinear (self contractions / Clebsch-Gordan couplings of $a_\ell$. ; for $\ell=2$ they are the eigenvalues of the associated symmetric traceless tenseor - energy + "prolate vs oblate". No single "size" scalar captures the shapes. 

In [6]:
# same energy, different shape : one modulus is not the shape
from spectrospherics import plot_same_energy

plot_same_energy().servable()
# plot_same_energy().show()   # <- in browser version

Column
    [0] Markdown(str)
    [1] Row
        [0] IntSlider(end=5, label='degree ℓ', name='degree ℓ', start=2, value=2)
        [1] FloatSlider(label='bend the second f..., name='bend the second f..., step=0.01, value=0.35)
    [2] Plotly(Figure)
    [3] LaTeX(str, renderer='mathjax', styles={'font-size': '15px'}, width=520)
    [4] Markdown(str)

In [7]:
# shape (2 moduli) vs orientation (3 angles), on a degree-2 field
from spectrospherics import plot_orbit_invariants

plot_orbit_invariants().servable()
# plot_orbit_invariants().show()   # <- in browser version

Column
    [0] Markdown(str)
    [1] Row
        [0] Column
            [0] FloatSlider(end=1.5, label='λ₁  (modulus 1)', name='λ₁  (modulus 1)', start=-1.5, step=0.01, value=1.0)
            [1] FloatSlider(end=1.5, label='λ₂  (modulus 2)', name='λ₂  (modulus 2)', start=-1.5, step=0.01, value=-0.3)
        [1] Column
            [0] FloatSlider(end=6.283185307179586, label='α  (group coordinate)', name='α  (group coordinate)', step=0.01)
            [1] FloatSlider(end=3.141592653589793, label='β  (group coordinate)', name='β  (group coordinate)', step=0.01)
            [2] FloatSlider(end=6.283185307179586, label='γ  (group coordinate)', name='γ  (group coordinate)', step=0.01)
    [2] Row
        [0] Plotly(Figure)
        [1] Column
            [0] LaTeX(str, renderer='mathjax', styles={'font-size': '15px'}, width=520)
            [1] Markdown(str)

The difficulties all point to the same fix: do not drive raw coefficients or an orientation-flavored chart; separate shape from orientation explicitly.

Design in shape space — parametrize by the 2ℓ-2 invariants (energy + shape), one control per genuinely distinct-shape direction. No orientation redundancy here.
Realize a canonical representative coefficient vector with that shape (fixed reference pose).
Aim it with a separate rotation (quaternion → D^ℓ), the ≤3 orientation numbers, applied last.

This keeps shape (payload) and orientation (nuisance) on independent, non-redundant controls, and isolates the stabilizer problem to step 3: if the designed shape happens to be symmetric, only the aiming step loses a knob; the shape controls stay clean.

## To summarize...

The 2D polar recipe works because a mode is a 2-vector, its rotation group is abelian, and its orbit is a circle labelled by a single radius — so modulus = shape and phase = orientation split perfectly. In 3D none of those hold: a degree is a (2ℓ+1)-vector; SO(3) is non-commutative so only yaw acts as a phase while tilt/roll mix the modes through the Wigner matrix; the shape is not one modulus but a small vector of nonlinear invariants; and the "redundancy" is that rotation only re-orients within a ≤3-dimensional orbit — so raw coefficient sliders waste up to three directions on aiming, and on symmetric shapes a rotation direction spins the shape into itself and a control goes dead (the stabilizer). The SH coefficients themselves are a clean, non-redundant basis; the permutation headaches come only from geometric charts laid on top. The practical escape is to design in the invariants and aim with a quaternion afterwards, keeping shape and orientation on separate, non-redundant controls.

In [8]:
# the whole argument in one picture : 2 = 1+1, but 2l+1 = (2l-2)+3
from spectrospherics import plot_polar_2d_vs_3d

plot_polar_2d_vs_3d().servable()
# plot_polar_2d_vs_3d().show()   # <- in browser version

Column
    [0] Markdown(str)
    [1] Row
        [0] IntSlider(end=5, label='2D : circular order n', name='2D : circular order n', start=1, value=3)
        [1] IntSlider(end=4, label='3D : degree ℓ', name='3D : degree ℓ', start=1, value=2)
        [2] FloatSlider(end=360, label='yaw α about z..., name='yaw α about z..., step=1, value=60)
        [3] FloatSlider(end=180, label='tilt β about y..., name='tilt β about y..., step=1)
    [2] Row
        [0] Plotly(Figure)
        [1] Column
            [0] LaTeX(str, renderer='mathjax', styles={'font-size': '15px'}, width=520)
            [1] Markdown(str)